# 1.0 Data Cleaning

Clean and normalize leads data fields.

In [ ]:
import re
from pathlib import Path
import pandas as pd

DATA_RAW = Path('../data/raw')
DATA_INTERIM = Path('../data/interim')
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_RAW / 'leads_seed.csv')
df.head()

## 1. Clean Lead Status

In [ ]:
df['lead_status_clean'] = df['Lead Status'].fillna('Unknown').astype(str).str.strip().str.title()
df['lead_status_clean'].value_counts()

## 2. Standardize Names

In [ ]:
def resolve_full_name(row):
    first = str(row['First Name']).strip() if pd.notna(row['First Name']) else ''
    last = str(row['Last Name']).strip() if pd.notna(row['Last Name']) else ''
    full = str(row['Full Name']).strip() if pd.notna(row['Full Name']) else ''
    if first or last:
        return f'{first} {last}'.strip()
    return full

df['name_clean'] = df.apply(resolve_full_name, axis=1)
df[['First Name', 'Last Name', 'Full Name', 'name_clean']].head()

## 3. Normalize Phone and Email

In [ ]:
df['phone_digits'] = df['Phone Number'].fillna('').astype(str).apply(lambda x: re.sub(r'\D', '', x))
df['email_clean'] = df['Email'].fillna('').astype(str).str.strip().str.lower()
df[['Phone Number', 'phone_digits', 'Email', 'email_clean']].head()

## 4. Parse Dates

In [ ]:
df['create_date_clean'] = pd.to_datetime(df['Create Date'], errors='coerce')
df['modified_date_clean'] = pd.to_datetime(df['Last Modified Date'], errors='coerce')
df[['Create Date', 'create_date_clean']].head()

## 5. Export Cleaned Interim Data

In [ ]:
output_file = DATA_INTERIM / 'leads_cleaned.csv'
df.to_csv(output_file, index=False)
print('Saved cleaned data to:', output_file)